<a href="https://colab.research.google.com/github/Mukundan-T/seqADAGE/blob/master/Py/muk_transfer_learning/genomic_mapping/Mukundan_sA_ec_pg_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E. coli &rarr; S. aureus Gene Mapping

### Mukundan Thanigaivelan

#### July 9, 2026

Here we are trying to map E. coli genes to S. aureus genes using techniques such as BLAST, Salmon, and Foldseek.

## 1. Connect to GitHub

In [1]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

Cloning into 'seqADAGE'...
remote: Enumerating objects: 603, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 603 (delta 58), reused 32 (delta 32), pack-reused 526 (from 2)
Receiving objects: 100% (603/603), 52.94 MiB | 32.25 MiB/s, done.
Resolving deltas: 100% (308/308), done.


In [2]:
%cd seqADAGE/Py/muk_transfer_learning/genomic_mapping

/content/seqADAGE/Py/muk_transfer_learning/genomic_mapping


## 2. Loading classes & modules

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [5]:
# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import tensorflow as tf
import os

In [6]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  1
True


In [7]:
ec_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa'
sa_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_pan_genome_reference.fa'

## 3. BLAST

### Installation

In [ ]:
!apt-get -qq update
!apt-get -qq install ncbi-blast+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!blastn -version

blastn: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


### Make BLAST database

In [ ]:
!makeblastdb -in "{ec_fasta}" -dbtype nucl -out ecoli_db



Building a new DB, current time: 07/20/2026 15:53:43
New DB name:   /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
New DB title:  /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa
Sequence type: Nucleotide
Deleted existing Nucleotide BLAST database named /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 68546 sequences in 1.5367 seconds.




### Blast S. aureus genes against database and save all hits

In [ ]:
blast_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_all_blast_hits.tsv'

In [ ]:
!blastn \
  -query "{sa_fasta}" \
  -db ecoli_db \
  -out "{blast_hits_file}" \
  -outfmt 6

In [ ]:
columns = [
  "query", "subject", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(blast_hits_file, sep = "\t", names = columns)
hits.shape

(7312, 12)

### EDA on BLAST hits

In [ ]:
hits.head()

,query,subject,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,12b493b74dec297122c11824ed05ba01_3,panDB562_47409,99.780,1362,3,0,1,1362,1,1362,0.0,2499.0
1,12b493b74dec297122c11824ed05ba01_5,panDB562_47408,99.647,1134,4,0,1,1134,1,1134,0.0,2073.0
2,12b493b74dec297122c11824ed05ba01_9,panDB562_47407,99.820,1113,2,0,1,1113,1,1113,0.0,2045.0
3,12b493b74dec297122c11824ed05ba01_11,panDB562_47406,99.845,1935,3,0,1,1935,1,1935,0.0,3557.0
4,12b493b74dec297122c11824ed05ba01_13,panDB562_47405,98.390,2670,43,0,1,2670,1,2670,0.0,4693.0


In [ ]:
hits.describe()

,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
count,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7.312000e+03,7312.000000
mean,97.386560,548.622812,11.576586,0.509847,126.768326,674.059354,216.755607,717.489606,9.927411e-09,945.319283
std,4.403544,524.543432,28.895761,2.625259,1198.731851,1302.197932,425.747971,593.130040,4.479023e-07,940.864001
min,72.301000,28.000000,0.000000,0.000000,1.000000,28.000000,1.000000,1.000000,0.000000e+00,52.800000
25%,97.143000,209.000000,1.000000,0.000000,1.000000,227.750000,1.000000,300.000000,0.000000e+00,344.000000
50%,99.142000,357.000000,3.000000,0.000000,1.000000,402.000000,1.000000,602.000000,4.800000e-166,582.000000
75%,99.822000,759.000000,10.000000,0.000000,1.000000,858.000000,271.000000,975.000000,1.775000e-94,1308.000000
max,100.000000,7176.000000,415.000000,69.000000,31333.000000,31635.000000,7012.000000,7341.000000,3.750000e-05,13075.000000


In [ ]:
# Number of unique SA genes represented
hits['query'].nunique()

4356

In [ ]:
# Number of unique EC genes represented
hits['subject'].nunique()

5255

In [ ]:
# See number of S. aureus genes that appear with a given frequency
hits['query'].value_counts().value_counts().head()

,count
count,
1,3621
2,276
3,99
4,81
5,78


In [ ]:
# See number of E. coli genes that appear with a given frequency
hits['subject'].value_counts().value_counts().head()

,count
count,
1,4398
2,388
3,227
4,127
5,61


### Filter for optimal best hits

In [ ]:
# Get all the S. aureus genes with only one hit
sa_genes_with_one_hit = hits[hits['query'].map(hits['query'].value_counts()) == 1]
sa_genes_with_one_hit.shape

(3621, 12)

In [ ]:
# Get all the S. aureus genes with many hits
sa_genes_with_many_hits = hits[hits['query'].map(hits['query'].value_counts()) != 1]
sa_genes_with_many_hits.shape

(3691, 12)

In [ ]:
### Code generated by ChatGPT 5.5 mini
import pulp

# Save the unique SA and EC genes to NumPy arrays
unique_ec_genes = sa_genes_with_many_hits['subject'].unique()
unique_sa_genes = sa_genes_with_many_hits['query'].unique()

# Create the optimization problem
prob = pulp.LpProblem("MaximizeUniqueSubjects", pulp.LpMaximize)

# One binary variable for each possible assignment (row)
x = {
  i: pulp.LpVariable(f"x_{i}", cat="Binary")
  for i in sa_genes_with_many_hits.index
}

# One binary variable for each subject
y = {
  s: pulp.LpVariable(f"y_{s}", cat="Binary")
  for s in unique_ec_genes
}

# ----------------------------
# Constraints
# ----------------------------

# Each query must choose exactly one subject
for q in unique_sa_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["query"] == q]
  prob += pulp.lpSum(x[i] for i in rows) == 1

# If a row uses a subject, that subject is "used"
for s in unique_ec_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["subject"] == s]
  for i in rows:
    prob += x[i] <= y[s]

# ----------------------------
# Objective
# ----------------------------

# Normalize e-values so the penalty is well behaved.
# If your evalues are already transformed (e.g. -log10),
# adjust this accordingly.

e = sa_genes_with_many_hits["evalue"].astype(float)

# Small penalty for worse evalues
eps = 1e-6

prob += (
  pulp.lpSum(y.values())
  - eps * pulp.lpSum(e[i] * x[i] for i in sa_genes_with_many_hits.index)
)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=True))

# Extract selected rows
result = sa_genes_with_many_hits[[x[i].value() > 0.5 for i in sa_genes_with_many_hits.index]].copy()
result.shape

(735, 12)

In [ ]:
print(
  f"There are {result['query'].nunique()} unique S. aureus genes and {result['subject'].nunique()} unique E. coli genes."
)

There are 735 unique S. aureus genes and 576 unique E. coli genes.


In [ ]:
# Now we have the best hit for each SA gene
best_hits = pd.concat([sa_genes_with_one_hit, result], ignore_index=True)
best_hits.shape

(4356, 12)

In [ ]:
# We've picked up 2,735 unique E. coli genes!
best_hits['subject'].nunique()

2735

In [ ]:
# Save best hits
best_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_best_blast_hits.csv',
  index = True
)

In [ ]:
# Verify I can load back in data
best_hits = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/BLAST/ec_sa_best_blast_hits.csv',
  index_col = 0
)
best_hits.shape

(4356, 12)

## 4a. $k$-mer mapping (Salmon)

### Installation

In [10]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda


In [11]:
# Add to PATH
os.environ["PATH"] += ":/usr/local/miniconda/bin"

In [12]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

In [13]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [14]:
!conda install -c bioconda -c conda-forge salmon

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ done
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: | / - done

## Package Plan ##

  environment location: /usr/local/miniconda

  added / updated specs:
    - salmon


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    certifi-2026.7.22          |     pyhd8ed1ab_0         134 KB  conda-forge
    conda-26.5.3               |  py314h9e666f3_2         1.4 MB  conda-forge
    openssl-3.6.3              |       h35e630c_0         3.0 MB  conda-forge
    salmon-2.3.4               |       hfa8f182_0         3.2 MB  bioconda
    ------------------------------------------------------------
                                           Total:         7.8 MB

The following NEW packages will be I

### Run Salmon and save hits

In [19]:
!salmon index \
    -t "{ec_fasta}" \
    -i ecoli_index

2026-07-22T21:21:40.082611Z  INFO salmon_index: replaced 3192 non-ACGT base(s) with random bases
2026-07-22T21:21:40.082640Z  INFO salmon_index: removed 9022 transcripts that were exact sequence duplicates (use --keepDuplicates to retain them)
2026-07-22T21:21:40.083176Z  INFO salmon_index: building compacted dBG (k=31, threads=8)
2026-07-22T21:21:40.087229Z  INFO cf1_rs::pipeline: RSS at start: current=72 MB, peak=1051 MB
2026-07-22T21:21:40.087250Z  INFO cf1_rs::pipeline: Phase 1: Counting minimizer frequencies...
2026-07-22T21:21:40.087254Z  INFO cf1_rs::minimizer: Counting minimizer frequencies with m=15, w=17, k=31
2026-07-22T21:21:40.519530Z  INFO cf1_rs::minimizer: Histogram: 4379280 total minimizer occurrences across 65536 non-empty buckets
2026-07-22T21:21:40.522419Z  INFO cf1_rs::pipeline: RSS after P1: current=90 MB, peak=1051 MB
2026-07-22T21:21:40.522433Z  INFO cf1_rs::pipeline: Phase 2: Partitioning minimizers and routing super k-mers...
2026-07-22T21:21:40.522511Z  INFO 

In [20]:
!salmon quant \
    -i ecoli_index \
    -l U \
    -r "{sa_fasta}" \
    --writeMappings='/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_all_kmer_hits.sam' \
    -o salmon_out

2026-07-22T21:23:07.794830Z  INFO piscem_rs::index::reference_index: Loading SSHash dictionary from ecoli_index/index.ssi
2026-07-22T21:23:07.819917Z  INFO piscem_rs::index::reference_index:   k=31, 258309 strings, canonical=true
2026-07-22T21:23:07.819976Z  INFO piscem_rs::index::reference_index: Loading contig table from ecoli_index/index.ctab
2026-07-22T21:23:07.820853Z  INFO piscem_rs::index::reference_index:   258309 contigs, 539302 entries, entry_width=31 bits
2026-07-22T21:23:07.820880Z  INFO piscem_rs::index::reference_index: Loading reference info from ecoli_index/index.refinfo
2026-07-22T21:23:07.824499Z  INFO piscem_rs::index::reference_index:   59524 references, max_ref_len=15876
2026-07-22T21:23:07.824555Z  INFO piscem_rs::index::reference_index: Loading equivalence class table from ecoli_index/index.ectab
2026-07-22T21:23:07.825241Z  INFO piscem_rs::index::reference_index:   258309 tiles, 139624 ECs, 309418 label entries
2026-07-22T21:23:07.825270Z  INFO piscem_rs::index:

In [33]:
sam_columns = [
    "QNAME", "FLAG", "RNAME", "POS", "MAPQ", "CIGAR",
    "RNEXT", "PNEXT", "TLEN", "SEQ", "QUAL"
]

kmer_hits = pd.read_csv(
    '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_all_kmer_hits.sam',
    sep="\t",
    comment="@", # Skip headers
    names=sam_columns,
    usecols=range(11),
    dtype=str
)
kmer_hits.shape

(4099, 11)

### EDA on hits

In [41]:
kmer_hits.head()

,QNAME,FLAG,RNAME,POS,MAPQ,CIGAR,RNEXT,PNEXT,TLEN,SEQ,QUAL
0,12b493b74dec297122c11824ed05ba01_3,0,panDB562_47409,1,1,1362M,*,0,0,ATGTCGGAAAAAGAAATTTGGGAAAAAGTGCTTGAAATTGCTCAAG...,*
1,12b493b74dec297122c11824ed05ba01_5,0,panDB562_47408,1,1,1134M,*,0,0,ATGATGGAATTCACTATTAAAAGAGATTATTTTATTACACAATTAA...,*
2,12b493b74dec297122c11824ed05ba01_9,0,panDB562_47407,1,1,1113M,*,0,0,ATGAAGTTAAATACACTCCAATTAGAAAATTATCGTAACTATGATG...,*
3,12b493b74dec297122c11824ed05ba01_11,0,panDB562_47406,1,1,1935M,*,0,0,ATGGTGACTGCATTGTCAGATGTAAACAACACGGATAATTATGGTG...,*
4,12b493b74dec297122c11824ed05ba01_13,0,panDB562_47405,1,1,2670M,*,0,0,ATGGCTGAATTACCTCAATCAAGAATAAATGAACGAAATATTACCA...,*


In [43]:
# Number of unique SA genes represented
kmer_hits['QNAME'].nunique()

3684

In [44]:
# Number of unique EC genes represented
kmer_hits['RNAME'].nunique()

2831

In [45]:
# See number of S. aureus genes that appear with a given frequency
kmer_hits['QNAME'].value_counts().value_counts().head()

,count
count,
1,3456
2,145
3,43
4,25
5,7


In [46]:
# See number of E. coli genes that appear with a given frequency
kmer_hits['RNAME'].value_counts().value_counts().head()

,count
count,
1,2264
3,208
2,205
4,101
5,42


### Filter for best hits

In [47]:
# Get all the S. aureus genes with only one hit
sa_genes_with_one_hit = kmer_hits[kmer_hits['QNAME'].map(kmer_hits['QNAME'].value_counts()) == 1]
sa_genes_with_one_hit.shape

(3456, 11)

In [48]:
# Get all the S. aureus genes with many hits
sa_genes_with_many_hits = kmer_hits[kmer_hits['QNAME'].map(kmer_hits['QNAME'].value_counts()) != 1]
sa_genes_with_many_hits.shape

(643, 11)

In [49]:
### Code generated by ChatGPT 5.5 mini
import pulp

# Save the unique SA and EC genes to NumPy arrays
unique_ec_genes = sa_genes_with_many_hits['RNAME'].unique()
unique_sa_genes = sa_genes_with_many_hits['QNAME'].unique()

# Create the optimization problem
prob = pulp.LpProblem("MaximizeUniqueSubjects", pulp.LpMaximize)

# One binary variable for each possible assignment (row)
x = {
  i: pulp.LpVariable(f"x_{i}", cat="Binary")
  for i in sa_genes_with_many_hits.index
}

# One binary variable for each subject
y = {
  s: pulp.LpVariable(f"y_{s}", cat="Binary")
  for s in unique_ec_genes
}

# ----------------------------
# Constraints
# ----------------------------

# Each query must choose exactly one subject
for q in unique_sa_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["QNAME"] == q]
  prob += pulp.lpSum(x[i] for i in rows) == 1

# If a row uses a subject, that subject is "used"
for s in unique_ec_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["RNAME"] == s]
  for i in rows:
    prob += x[i] <= y[s]

# ----------------------------
# Objective
# ----------------------------

prob += pulp.lpSum(y.values())

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=True))

# Extract selected rows
result = sa_genes_with_many_hits[[x[i].value() > 0.5 for i in sa_genes_with_many_hits.index]].copy()
result.shape

(228, 11)

In [50]:
print(
  f"There are {result['QNAME'].nunique()} unique S. aureus genes and {result['RNAME'].nunique()} unique E. coli genes."
)

There are 228 unique S. aureus genes and 212 unique E. coli genes.


In [51]:
# Now we have the best hit for each SA gene
best_hits = pd.concat([sa_genes_with_one_hit, result], ignore_index=True)
best_hits.shape

(3684, 11)

In [54]:
# We've picked up 2,439 unique E. coli genes!
best_hits['RNAME'].nunique()

2439

In [55]:
# Save best hits
best_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_best_kmer_hits.csv',
  index = True
)

In [56]:
# Verify I can load back in data
best_hits = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_best_kmer_hits.csv',
  index_col = 0
)
best_hits.shape

(3684, 11)

## 4b. $k$-mer mapping (BLAT)

### Installation

In [ ]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

ERROR: File or directory already exists: '/usr/local/miniconda'
If you want to update an existing installation, use the -u option.


In [ ]:
# Add to PATH
os.environ["PATH"] += ":/usr/local/miniconda/bin"

In [ ]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [ ]:
!conda install -y blat

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: - \ | done

# All requested packages already installed.



In [ ]:
!blat

blat - Standalone BLAT v. 35 fast sequence search command line tool
usage:
   blat database query [-ooc=11.ooc] output.psl
where:
   database and query are each either a .fa , .nib or .2bit file,
   or a list these files one file name per line.
   -ooc=11.ooc tells the program to load over-occurring 11-mers from
               and external file.  This will increase the speed
               by a factor of 40 in many cases, but is not required
   output.psl is where to put the output.
   Subranges of nib and .2bit files may specified using the syntax:
      /path/file.nib:seqid:start-end
   or
      /path/file.2bit:seqid:start-end
   or
      /path/file.nib:start-end
   With the second form, a sequence id of file:start-end will be used.
options:
   -t=type     Database type.  Type is one of:
                 dna - DNA sequence
                 prot - protein sequence
                 dnax - DNA sequence translated in six frames to protein
               The default is dna
   -q=type     

### Run BLAT and save all hits

In [ ]:
# File to store k-mer best hits
kmer_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_all_kmer_hits.psl'

In [ ]:
!blat \
  "{ec_fasta}" \
  "{sa_fasta}" \
  "{kmer_hits_file}"

Loaded 46157027 letters in 68546 sequences
Searched 6144633 bases in 9935 sequences


In [ ]:
psl_columns = [
  "matches", "misMatches", "repMatches", "nCount",
  "qNumInsert", "qBaseInsert", "tNumInsert", "tBaseInsert",
  "strand", "qName", "qSize", "qStart", "qEnd", "tName",
  "tSize", "tStart", "tEnd", "blockCount", "blockSizes",
  "qStarts", "tStarts",
]

blat_hits = pd.read_csv(
  kmer_hits_file,
  sep = "\t",
  skiprows = 5,
  names = psl_columns,
)
blat_hits.shape

(7030, 21)

### EDA on $k$-mer hits

In [ ]:
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,qStart,qEnd,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,0,1362,panDB562_47409,1362,0,1362,1,"1362,","0,","0,"
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,0,1134,panDB562_47408,1134,0,1134,1,"1134,","0,","0,"
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,0,1113,panDB562_47407,1113,0,1113,1,"1113,","0,","0,"
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,0,1935,panDB562_47406,1935,0,1935,1,"1935,","0,","0,"
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,0,2666,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,"


In [ ]:
blat_hits.describe()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,qSize,qStart,qEnd,tSize,tStart,tEnd,blockCount
count,7030.000000,7030.000000,7030.0,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000,7030.000000
mean,532.342959,6.440825,0.0,0.006117,0.082930,3.750213,0.102276,5.615932,697.577098,99.989616,642.529730,811.418777,190.939829,735.345661,1.137269
std,527.244355,11.481178,0.0,0.132142,0.447667,44.863219,0.534679,41.583894,1320.500307,795.313384,946.004964,625.160113,403.949294,578.639056,0.664372
min,30.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,70.000000,0.000000,32.000000,63.000000,0.000000,32.000000,1.000000
25%,198.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,234.000000,0.000000,219.000000,369.000000,0.000000,315.000000,1.000000
50%,331.500000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,429.000000,0.000000,390.000000,675.000000,0.000000,609.000000,1.000000
75%,744.000000,8.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,888.000000,0.000000,854.500000,1062.000000,203.250000,981.000000,1.000000
max,7199.000000,296.000000,0.0,8.000000,11.000000,2073.000000,10.000000,799.000000,31635.000000,29478.000000,31497.000000,7341.000000,6095.000000,7341.000000,15.000000


In [ ]:
# Number of unique SA genes represented
blat_hits['qName'].nunique()

4246

In [ ]:
# Number of unique EC genes represented
blat_hits['tName'].nunique()

5171

In [ ]:
# See number of S. aureus genes that appear with a given frequency
blat_hits['qName'].value_counts().value_counts().head()

,count
count,
1,3578
2,238
4,85
3,83
5,73


In [ ]:
# See number of E. coli genes that appear with a given frequency
blat_hits['tName'].value_counts().value_counts().head()

,count
count,
1,4367
2,359
3,230
4,126
5,52


### Add custom scoring

In [ ]:
# Define a percentage match metric
blat_hits['pident'] = 100 * (blat_hits['matches'] / (blat_hits['matches'] + blat_hits['misMatches']))

# Custom score since we don't necessarily get a 'pident' --> reward high number of matches and high percentage of matches
blat_hits['score'] = blat_hits['matches'] * (blat_hits['pident'] / 100)
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts,pident,score
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,panDB562_47409,1362,0,1362,1,"1362,","0,","0,",99.779736,1356.006608
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,panDB562_47408,1134,0,1134,1,"1134,","0,","0,",99.647266,1126.014109
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,panDB562_47407,1113,0,1113,1,"1113,","0,","0,",99.820305,1109.003594
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,panDB562_47406,1935,0,1935,1,"1935,","0,","0,",99.844961,1929.004651
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,",98.420459,2575.663407


### Filter for optimal best hits

In [ ]:
# Get all the S. aureus genes with only one hit
sa_genes_with_one_hit = blat_hits[blat_hits['qName'].map(blat_hits['qName'].value_counts()) == 1]
sa_genes_with_one_hit.shape

(3578, 23)

In [ ]:
# Get all the S. aureus genes with many hits
sa_genes_with_many_hits = blat_hits[blat_hits['qName'].map(blat_hits['qName'].value_counts()) != 1]
sa_genes_with_many_hits.shape

(3452, 23)

In [ ]:
### Code generated by ChatGPT 5.5 mini
import pulp

# Save the unique SA and EC genes to NumPy arrays
unique_ec_genes = sa_genes_with_many_hits['tName'].unique()
unique_sa_genes = sa_genes_with_many_hits['qName'].unique()

# Create the optimization problem
prob = pulp.LpProblem("MaximizeUniqueSubjects", pulp.LpMaximize)

# One binary variable for each possible assignment (row)
x = {
  i: pulp.LpVariable(f"x_{i}", cat="Binary")
  for i in sa_genes_with_many_hits.index
}

# One binary variable for each subject
y = {
  s: pulp.LpVariable(f"y_{s}", cat="Binary")
  for s in unique_ec_genes
}

# ----------------------------
# Constraints
# ----------------------------

# Each query must choose exactly one subject
for q in unique_sa_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["qName"] == q]
  prob += pulp.lpSum(x[i] for i in rows) == 1

# If a row uses a subject, that subject is "used"
for s in unique_ec_genes:
  rows = sa_genes_with_many_hits.index[sa_genes_with_many_hits["tName"] == s]
  for i in rows:
    prob += x[i] <= y[s]
  prob += y[s] <= pulp.lpSum(x[i] for i in rows)

# ----------------------------
# Objective
# ----------------------------

score = sa_genes_with_many_hits['score'].astype(float)

# Make sure one additional unique subject is always worth more
# than any possible score difference.
score_weight = 1e-6

prob += (
    pulp.lpSum(y.values())
    + score_weight * pulp.lpSum(score[i] * x[i] for i in sa_genes_with_many_hits.index)
)

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=True))

# Extract selected rows
result = sa_genes_with_many_hits[[x[i].value() > 0.5 for i in sa_genes_with_many_hits.index]].copy()
result.shape

(668, 23)

In [ ]:
print(
  f"There are {result['qName'].nunique()} unique S. aureus genes and {result['tName'].nunique()} unique E. coli genes."
)

There are 668 unique S. aureus genes and 549 unique E. coli genes.


In [ ]:
# Now we have the best hit for each SA gene
best_hits = pd.concat([sa_genes_with_one_hit, result], ignore_index=True)
best_hits.shape

(4246, 23)

In [ ]:
# We've picked up 2,731 unique E. coli genes!
best_hits['tName'].nunique()

2731

In [ ]:
# Save best hits
best_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_best_kmer_hits.csv',
  index = True
)

In [ ]:
# Verify I can load back in data
best_hits = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/k-mer-mapping/ec_sa_best_kmer_hits.csv',
  index_col = 0
)
best_hits.shape

(4246, 23)

## 5. Foldseek

### Convert DNA FASTA to Protein FASTA

In [57]:
!pip install -qq biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.6 MB/s eta 0:00:00


In [67]:
ec_prot_fasta = "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_proteins.faa"
sa_prot_fasta = "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/sa_proteins.faa"

In [68]:
from Bio import SeqIO

def cds_to_protein(input_fasta, output_fasta):
  """
  Given an input FASTA file with Coding Sequences (CDS), convert it
  to a Protein FASTA.
  """
  proteins = []

  for record in SeqIO.parse(input_fasta, "fasta"):
    protein_seq = record.seq.translate(table = 11, to_stop = True)
    record.seq = protein_seq
    proteins.append(record)

  SeqIO.write(proteins, output_fasta, "fasta")

cds_to_protein(ec_fasta, ec_prot_fasta)
cds_to_protein(sa_fasta, sa_prot_fasta)

/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


### Install Foldseek

In [100]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

ERROR: File or directory already exists: '/usr/local/miniconda'
If you want to update an existing installation, use the -u option.


In [101]:
# Add to PATH
os.environ["PATH"] += ":/usr/local/miniconda/bin"

In [102]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

In [103]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [104]:
!conda install -c bioconda foldseek

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ | / done

# All requested packages already installed.



In [105]:
!foldseek version

10.941cd33


### Create Foldseek databases

In [106]:
!foldseek databases ProstT5 prostt5_out tmp

prostt5_out exists and will be overwritten
databases ProstT5 prostt5_out tmp 

MMseqs Version:              	10.941cd33
Tsv                          	false
Force restart with latest tmp	false
Remove temporary files       	false
Compressed                   	0
Threads                      	8
Verbosity                    	3

prostt5-f16.gguf


In [111]:
# Create S. aureus database
!foldseek createdb "{sa_prot_fasta}" saur_pan_genome_dbg --prostt5-model prostt5_out --gpu 1

saur_pan_genome_dbg exists and will be overwritten
saur_pan_genome_dbg exists and will be overwritten
createdb /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/sa_proteins.faa saur_pan_genome_dbg --prostt5-model prostt5_out --gpu 1 

MMseqs Version:             	10.941cd33
Use GPU                     	1
Path to ProstT5             	prostt5_out
Chain name mode             	0
Createdb extraction mode    	0
Interface distance threshold	8
Write mapping file          	0
Mask b-factor threshold     	0
Coord store mode            	2
Write lookup file           	1
Input format                	0
File Inclusion Regex        	.*
File Exclusion Regex        	^$
Threads                     	8
Verbosity                   	3

Converting sequences
[9899] 0s 21ms
Time for merging to saur_pan_genome_dbg_h: 0h 0m 0s 3ms
Time for merging to saur_pan_genome_dbg: 0h 0m 0s 5ms
Database type: Aminoacid
CUDA0
CPU
[=================================================================] 100.00% 9.94

In [115]:
# Create E. coli database
!foldseek createdb "{ec_prot_fasta}" ec_pan_genome_dbg --prostt5-model prostt5_out --gpu 1

ec_pan_genome_dbg exists and will be overwritten
ec_pan_genome_dbg exists and will be overwritten
createdb /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_proteins.faa ec_pan_genome_dbg --prostt5-model prostt5_out --gpu 1 

MMseqs Version:             	10.941cd33
Use GPU                     	1
Path to ProstT5             	prostt5_out
Chain name mode             	0
Createdb extraction mode    	0
Interface distance threshold	8
Write mapping file          	0
Mask b-factor threshold     	0
Coord store mode            	2
Write lookup file           	1
Input format                	0
File Inclusion Regex        	.*
File Exclusion Regex        	^$
Threads                     	8
Verbosity                   	3

Converting sequences
[68479] 0s 89ms
Time for merging to ec_pan_genome_dbg_h: 0h 0m 0s 11ms
Time for merging to ec_pan_genome_dbg: 0h 0m 0s 24ms
Database type: Aminoacid
CUDA0
CPU
[=================================================================] 100.00% 68.55K 2h 1

In [116]:
# Make S. aureus padded database
!foldseek makepaddedseqdb saur_pan_genome_dbg saur_pan_genome_dbg_pad

saur_pan_genome_dbg_pad exists and will be overwritten
makepaddedseqdb saur_pan_genome_dbg saur_pan_genome_dbg_pad 

MMseqs Version:          	10.941cd33
Substitution matrix      	aa:3di.out,nucl:3di.out
Mask residues            	0
Mask residues probability	0.999995
Write lookup file        	1
Threads                  	8
Verbosity                	3
Cluster search           	0

lndb saur_pan_genome_dbg_h saur_pan_genome_dbg_pad_tmp_ss_h 

Time for processing: 0h 0m 0s 0ms
lndb saur_pan_genome_dbg_ss saur_pan_genome_dbg_pad_tmp_ss 

Time for processing: 0h 0m 0s 0ms
makepaddedseqdb saur_pan_genome_dbg_pad_tmp_ss saur_pan_genome_dbg_pad_ss --sub-mat 'aa:3di.out,nucl:3di.out' --score-bias 0 --mask 0 --mask-prob 0.999995 --mask-lower-case 1 --mask-n-repeat 6 --write-lookup 1 --threads 8 -v 3 

[=================================================================] 100.00% 9.94K 0s 190ms
Time for merging to saur_pan_genome_dbg_pad_ss: 0h 0m 0s 3ms
Time for merging to saur_pan_genome_dbg_pad_ss_h

In [117]:
# Make E. coli padded database
!foldseek makepaddedseqdb ec_pan_genome_dbg ec_pan_genome_dbg_pad

ec_pan_genome_dbg_pad exists and will be overwritten
makepaddedseqdb ec_pan_genome_dbg ec_pan_genome_dbg_pad 

MMseqs Version:          	10.941cd33
Substitution matrix      	aa:3di.out,nucl:3di.out
Mask residues            	0
Mask residues probability	0.999995
Write lookup file        	1
Threads                  	8
Verbosity                	3
Cluster search           	0

lndb ec_pan_genome_dbg_h ec_pan_genome_dbg_pad_tmp_ss_h 

Time for processing: 0h 0m 0s 0ms
lndb ec_pan_genome_dbg_ss ec_pan_genome_dbg_pad_tmp_ss 

Time for processing: 0h 0m 0s 0ms
makepaddedseqdb ec_pan_genome_dbg_pad_tmp_ss ec_pan_genome_dbg_pad_ss --sub-mat 'aa:3di.out,nucl:3di.out' --score-bias 0 --mask 0 --mask-prob 0.999995 --mask-lower-case 1 --mask-n-repeat 6 --write-lookup 1 --threads 8 -v 3 

[=================================================================] 100.00% 68.55K 0s 322ms
Time for merging to ec_pan_genome_dbg_pad_ss: 0h 0m 0s 20ms
Time for merging to ec_pan_genome_dbg_pad_ss_h: 0h 0m 0s 8ms
Time 

### Save Foldseek Databases

In [120]:
# Save E. coli database files to drive
!cp ec_pan_genome_dbg* "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/"

In [121]:
# Save S. aureus database files to drive
!cp saur_pan_genome_dbg* "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/"

### Run Foldseek search

In [118]:
!foldseek easy-search \
  saur_pan_genome_dbg_pad \
  ec_pan_genome_dbg_pad \
  "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_sa_all_foldseek_hits.m8" \
  tmpFolder \
  --gpu 1 \
  --exhaustive-search \
  -e 0.05 \
  --num-iterations 3

easy-search saur_pan_genome_dbg_pad ec_pan_genome_dbg_pad /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_sa_all_foldseek_hits.m8 tmpFolder --gpu 1 --exhaustive-search -e 0.05 --num-iterations 3 

MMseqs Version:                    	10.941cd33
Seq. id. threshold                 	0
Coverage threshold                 	0
Coverage mode                      	0
Max reject                         	2147483647
Max accept                         	2147483647
Add backtrace                      	false
TMscore threshold                  	0
TMscore threshold mode             	0
TMalign hit order                  	0
TMalign fast                       	1
Preload mode                       	0
Threads                            	8
Verbosity                          	3
LDDT threshold                     	0
Sort by structure bit score        	1
Alignment type                     	2
Exact TMscore                      	0
Substitution matrix                	aa:3di.out,nucl:3di.out
Alignm

### EDA on Foldseek hits

In [122]:
cols = [
  "query", "target", "pident", "alnlen", "mismatch", "gapopen",
  "qstart", "qend", "tstart", "tend", "evalue", "bitscore"
]

fs_hits = pd.read_csv(
    "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/Foldseek/ec_sa_all_foldseek_hits.m8",
    sep = "\t",
    header = None,
    names = cols
)
fs_hits.shape

(7384485, 12)

In [123]:
fs_hits.head()

,query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bitscore
0,12b493b74dec297122c11824ed05ba01_875,panDB562_39050,0.450,20,11,0,2,21,82,101,0.04318,92
1,12b493b74dec297122c11824ed05ba01_875,panDB562_38799,0.450,20,11,0,2,21,82,101,0.04318,92
2,9ab1425e5eb2f9df6f6be6d193592bac_2598,panDB562_49224,0.440,25,14,0,1,25,1,25,0.04501,135
3,9ab1425e5eb2f9df6f6be6d193592bac_2598,11997cc26382b2c286cd502685a104a5_5243,0.478,23,12,0,3,25,4,26,0.04785,134
4,12b493b74dec297122c11824ed05ba01_1731,panDB562_56318,0.444,27,11,1,4,26,512,538,0.04970,98


In [126]:
fs_hits.describe()

,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bitscore
count,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06,7.384485e+06
mean,1.031220e-01,2.271436e+02,1.361068e+02,1.147433e+01,4.862411e+01,2.394850e+02,7.793214e+01,2.699097e+02,3.703813e-03,3.159432e+02
std,5.544168e-02,1.485267e+02,8.153283e+01,8.340501e+00,1.936134e+02,2.436825e+02,1.407235e+02,1.753631e+02,9.253295e-03,2.030169e+02
min,0.000000e+00,1.400000e+01,0.000000e+00,0.000000e+00,1.000000e+00,1.600000e+01,1.000000e+00,1.800000e+01,0.000000e+00,8.900000e+01
25%,7.200000e-02,1.110000e+02,7.300000e+01,5.000000e+00,2.000000e+00,1.100000e+02,3.000000e+00,1.520000e+02,2.083000e-07,1.920000e+02
50%,9.100000e-02,1.930000e+02,1.170000e+02,1.000000e+01,8.000000e+00,1.960000e+02,2.200000e+01,2.340000e+02,4.779000e-05,2.630000e+02
75%,1.200000e-01,3.060000e+02,1.770000e+02,1.600000e+01,4.100000e+01,3.120000e+02,9.700000e+01,3.480000e+02,9.336000e-04,3.750000e+02
max,1.000000e+00,3.786000e+03,1.807000e+03,2.350000e+02,1.042700e+04,1.054400e+04,5.054000e+03,5.265000e+03,5.000000e-02,2.101000e+04


In [127]:
# Number of unique SA genes represented
fs_hits['query'].nunique()

9748

In [128]:
# Number of unique EC genes represented
fs_hits['target'].nunique()

59894

In [129]:
# See number of S. aureus genes that appear with a given frequency
fs_hits['query'].value_counts().value_counts().head()

,count
count,
1,60
2,51
1284,47
8,39
3,39


In [130]:
# See number of E. coli genes that appear with a given frequency
fs_hits['target'].value_counts().value_counts().head()

,count
count,
1,2193
2,1524
3,1219
4,1136
5,902


### Filter for optimal best hits

In [131]:
# Get all the S. aureus genes with only one hit
sa_genes_with_one_hit = fs_hits[fs_hits['query'].map(fs_hits['query'].value_counts()) == 1]
sa_genes_with_one_hit.shape

(60, 12)

In [134]:
# Get all the S. aureus genes with many hits
sa_genes_with_many_hits = fs_hits[fs_hits['query'].map(fs_hits['query'].value_counts()) != 1]
sa_genes_with_many_hits.shape

(7384425, 12)